In [1]:
!pip install SoccerNet --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.9/86.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 8.0 MB/s eta 0:00:00


In [2]:
from SoccerNet.Downloader import SoccerNetDownloader

In [ ]:
from SoccerNet.Downloader import SoccerNetDownloader

mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory="data")

mySoccerNetDownloader.password = "s0cc3rn3t"

mySoccerNetDownloader.downloadDataTask(
    task="spotting"
)

mySoccerNetDownloader.downloadDataTask(
    task="tracking"
)

mySoccerNetDownloader.downloadDataTask(
    task="reid"
)

In [ ]:
import numpy as np
import os

class SoccerNetMultiTaskLoader:
    def __init__(self, root):
        self.root = root

    def load_spotting(self, split="train"):
        X = np.load(os.path.join(self.root, "spotting", split, "features.npy"))
        y = np.load(os.path.join(self.root, "spotting", split, "labels.npy"))
        return X, y

    def load_tracking(self, split="train"):
        X = np.load(os.path.join(self.root, "tracking", split, "features.npy"))
        y = np.load(os.path.join(self.root, "tracking", split, "tracks.npy"))
        return X, y

    def load_reid(self, split="train"):
        X = np.load(os.path.join(self.root, "reid", split, "features.npy"))
        y = np.load(os.path.join(self.root, "reid", split, "ids.npy"))
        return X, y

In [ ]:
import torch
import torch.nn as nn

class TransformerBackbone(nn.Module):
    def __init__(self, input_dim=1024, d_model=256, nhead=8, num_layers=4):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        return x



class SpottingHead(nn.Module):
    def __init__(self, d_model=256, num_classes=10):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.fc(x)


class TrackingHead(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, 4)  # bbox
        )

    def forward(self, x):
        return self.fc(x)



class ReIDHead(nn.Module):
    def __init__(self, d_model=256, num_ids=100):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, num_ids)
        )

    def forward(self, x):
        return self.fc(x)


class MatchMindAI(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = TransformerBackbone()

        self.spotting = SpottingHead()
        self.tracking = TrackingHead()
        self.reid = ReIDHead()

    def forward(self, x):
        features = self.backbone(x)

        return {
            "spotting": self.spotting(features),
            "tracking": self.tracking(features),
            "reid": self.reid(features)
        }

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

from dataset_loader import SoccerNetMultiTaskLoader
from model import MatchMindAI

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loader = SoccerNetMultiTaskLoader("data")

X_s, y_s = loader.load_spotting("train")
X_t, y_t = loader.load_tracking("train")
X_r, y_r = loader.load_reid("train")


X = torch.tensor(X_s, dtype=torch.float32)

y_s = torch.tensor(y_s, dtype=torch.long)
y_t = torch.tensor(y_t, dtype=torch.float32)
y_r = torch.tensor(y_r, dtype=torch.long)

dataset = TensorDataset(X, y_s, y_t, y_r)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

model = MatchMindAI().to(device)

loss_spot = nn.CrossEntropyLoss()
loss_track = nn.SmoothL1Loss()
loss_reid = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(10):
    total_loss = 0

    for x, ys, yt, yr in dataloader:
        x = x.to(device)
        ys = ys.to(device)
        yt = yt.to(device)
        yr = yr.to(device)

        out = model(x)

        l1 = loss_spot(out["spotting"], ys)
        l2 = loss_track(out["tracking"], yt)
        l3 = loss_reid(out["reid"], yr)

        loss = l1 + l2 + l3

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f}")

torch.save(model.state_dict(), "matchmind_transformer.pth")

In [ ]:
import torch
from sklearn.metrics import f1_score
from model import MatchMindAI

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MatchMindAI().to(device)
model.load_state_dict(torch.load("matchmind_transformer.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x, ys, yt, yr in dataloader:
        x = x.to(device)

        out = model(x)

        preds = torch.argmax(out["spotting"], dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(ys.numpy())

print("F1 Score:", f1_score(all_labels, all_preds, average="macro"))